In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import urllib.request
from konlpy.tag import Mecab
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter

## 1. 네이버 영화 리뷰 데이터에 대한 이해와 전처리
- 200,000개 리뷰로 구성된 데이터
- 영화 리뷰에 대한 텍스트와 해당 리뷰가 긍정인 경우 1, 부정인 경우 0을 표시한 레이블로 구성

### 1.1 데이터 로드하기

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="data/ratings_train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt", filename="data/ratings_test.txt")

In [ ]:
train_data = pd.read_table('data/ratings_train.txt')
test_data = pd.read_table('data/ratings_test.txt')

In [ ]:
print('훈련용 리뷰 개수 :',len(train_data)) # 훈련용 리뷰 개수 출력

In [ ]:
train_data[:10] # 상위 10개 출력

In [ ]:
print('테스트용 리뷰 개수 :',len(test_data)) # 테스트용 리뷰 개수 출력

In [ ]:
test_data[:10] # 상위 10개 출력

### 1.2 데이터 정제하기
#### 1. 21 중복값 및 Null값 제거하기

In [ ]:
# document 열과 label 열의 중복을 제외한 값의 개수
train_data['document'].nunique(), train_data['label'].nunique()

In [ ]:
# document 열의 중복 제거
train_data.drop_duplicates(subset=['document'], inplace=True)

In [ ]:
print('총 샘플의 수 :',len(train_data))

In [ ]:
train_data['label'].value_counts().plot(kind = 'bar')

In [ ]:
print(train_data.groupby('label').size().reset_index(name = 'count'))

In [ ]:
print(train_data.isnull().sum())

In [ ]:
train_data.loc[train_data.document.isnull()]

In [ ]:
train_data = train_data.dropna(how = 'any') # Null 값이 존재하는 행 제거
print(train_data.isnull().values.any()) # Null 값이 존재하는지 확인

In [ ]:
print(len(train_data))

#### 1.22 데이터 전처리

In [ ]:
#알파벳과 공백을 제외하고 모두 제거
eng_text = 'do!!! you expect... people~ to~ read~ the FAQ, etc. and actually accept hard~! atheism?@@'
print(re.sub(r'[^a-zA-Z ]', '', eng_text))

[유니코드 한글 자모 문서](https://www.unicode.org/charts/PDF/U3130.pdf) <br>
[유니코드 한글 음절 문서](https://www.unicode.org/charts/PDF/UAC00.pdf)

In [ ]:
# [^0]처럼 ^ 문자가 대괄호 안에 들어 있다면, ^는 시작하지 않는다는 뜻
# 즉, 아래 조건은 한글과 공백으로 시작하지 않는 문자열을 의미함
train_data['document'] = train_data['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","", regex=True)
train_data[:10]

In [ ]:
# ^는 문자열의 시작을 의미
# 즉, 아래 조건은 white space로 시작하는 문자열을 의미함
train_data['document'] = train_data['document'].str.replace('^ +', "", regex=True) # white space 데이터를 empty value로 변경
train_data.replace({'document': ''}, np.nan, inplace=True)
print(train_data.isnull().sum())

train_data.loc[train_data.document.isnull()][:5]

In [ ]:
train_data = train_data.dropna(how = 'any')
print(len(train_data))

In [ ]:
test_data.drop_duplicates(subset = ['document'], inplace=True) # document 열에서 중복인 내용이 있다면 중복 제거
test_data['document'] = test_data['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","") # 정규 표현식 수행
test_data['document'] = test_data['document'].str.replace('^ +', "") # 공백은 empty 값으로 변경
test_data.replace({'document': ''}, np.nan, inplace=True) # 공백은 Null 값으로 변경
test_data = test_data.dropna(how='any') # Null 값 제거
print('전처리 후 테스트용 샘플의 개수 :',len(test_data))

#### 1.23 토큰화

In [ ]:
stopwords = ['도', '는', '다', '의', '가', '이', '은', '한', '에', '하', '고', '을', '를', '인', '듯', '과', '와', '네', '들', '듯', '지', '임', '게']

In [ ]:
mecab = Mecab()
X_train = []
for sentence in tqdm(train_data['document']):
    tokenized_sentence = mecab.morphs(sentence) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    X_train.append(stopwords_removed_sentence)

In [ ]:
print(X_train[:3])

In [ ]:
X_test = []
for sentence in tqdm(test_data['document']):
    tokenized_sentence = mecab.morphs(sentence) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    X_test.append(stopwords_removed_sentence)

#### 1.24 학습 데이터, 검증 데이터, 테스트 데이터
- 학습 데이터 중에서 20%를 분할하여 추가로 검증 데이터를 만듦
- 레이블의 균형 비율을 유지하면서 분할하고 싶다면, y데이터를 stratify의 값으로 사용

In [ ]:
y_train = np.array(train_data['label'])
y_test = np.array(test_data['label'])

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=0, stratify=y_train)

In [ ]:
print('--------훈련 데이터의 비율-----------')
print(f'부정 리뷰 = {round(np.sum(y_train==0)/len(y_train) * 100,3)}%')
print(f'긍정 리뷰 = {round(np.count_nonzero(y_train)/len(y_train) * 100,3)}%')
print('--------검증 데이터의 비율-----------')
print(f'부정 리뷰 = {round(np.sum(y_valid==0)/len(y_valid) * 100,3)}%')
print(f'긍정 리뷰 = {round(np.count_nonzero(y_valid)/len(y_valid) * 100,3)}%')
print('--------테스트 데이터의 비율-----------')
print(f'부정 리뷰 = {round(np.sum(y_test==0)/len(y_test) * 100,3)}%')
print(f'긍정 리뷰 = {round(np.count_nonzero(y_test)/len(y_test) * 100,3)}%')

#### 1.25 단어 집합 만들기
- 텍스트를 숫자로 처리할 수 있도록 훈련 데이터와 테스트 데이터에 정수 인코딩을 수행

In [ ]:
word_list = []
for sentence in X_train:
    for word in sentence:
      word_list.append(word)

word_counts = Counter(word_list)
print('총 단어수 :', len(word_counts))

In [ ]:
print('훈련 데이터에서의 단어 영화의 등장 횟수 :', word_counts['영화'])
print('훈련 데이터에서의 단어 공감의 등장 횟수 :', word_counts['공감'])

In [ ]:
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
print('등장 빈도수 상위 10개 단어')
print(vocab[:10])

- 빈도수가 낮은 단어들은 자연어 처리에서 배제하고자 합

In [ ]:
threshold = 3
total_cnt = len(word_counts) # 단어의 수
rare_cnt = 0 # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0 # 훈련 데이터의 전체 단어 빈도수 총 합
rare_freq = 0 # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수의 총 합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in word_counts.items():
    total_freq = total_freq + value

    # 단어의 등장 빈도수가 threshold보다 작으면
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 미만인 희귀 단어의 수: %s'%(threshold, rare_cnt))
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)

In [ ]:
# 전체 단어 개수 중 빈도수 2이하인 단어는 제거.
vocab_size = total_cnt - rare_cnt
vocab = vocab[:vocab_size] # 빈도수 기반 인덱싱을 적용하였으므로 가능
print('단어 집합의 크기 :', len(vocab))

In [ ]:
word_to_index = {}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

In [ ]:
for index, word in enumerate(vocab) :
  word_to_index[word] = index + 2

In [ ]:
vocab_size = len(word_to_index)
print('패딩 토큰과 UNK 토큰을 고려한 단어 집합의 크기 :', vocab_size)

In [ ]:
for i, word_index in enumerate(word_to_index.items()):
    print(f'단어 {word_index[0]}와 맵핑되는 정수 :{word_index[1]}')
    if i == 5: break

#### 1.26 정수 인코딩
- 단어 집합에 존재하지 않는 단어들은 일괄로 `<UNK>`(인덱스 1)로 맵핑

In [ ]:
def texts_to_sequences(tokenized_X_data, word_to_index):
  encoded_X_data = []
  for sentence in tokenized_X_data:
    index_sequences = []
    for word in sentence:
      try:
          index_sequences.append(word_to_index[word])
      except KeyError: # 단어가 존재하지 않는 경우
          index_sequences.append(word_to_index['<UNK>'])
    encoded_X_data.append(index_sequences)
  return encoded_X_data

In [ ]:
encoded_X_train = texts_to_sequences(X_train, word_to_index)
encoded_X_valid = texts_to_sequences(X_valid, word_to_index)
encoded_X_test = texts_to_sequences(X_test, word_to_index)

In [ ]:
# 상위 샘플 2개 출력
for sentence in encoded_X_train[:2]:
  print(sentence)

In [ ]:
index_to_word = {}
for key, value in word_to_index.items():
    index_to_word[value] = key

In [ ]:
decoded_sample = [index_to_word[word] for word in encoded_X_train[0]]
print('인코딩 전 첫번째 샘플 :', X_train[0])
print('인코딩 후 복원된 첫번째 샘플 :', decoded_sample)

#### 1.27 패딩
- 서로 다른 길이의 샘플들의 길이를 동일하게 맞춰주는 작업

In [ ]:
print('리뷰의 최대 길이 :',max(len(review) for review in encoded_X_train))
print('리뷰의 평균 길이 :',sum(map(len, encoded_X_train))/len(encoded_X_train))
plt.hist([len(review) for review in encoded_X_train], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
def below_threshold_len(max_len, nested_list):
  count = sum([1 for sentence in nested_list if len(sentence) <= max_len])
  print(f'전체 샘플 중 길이가 {max_len} 이하인 샘플의 비율: {(count / len(nested_list))*100: 0.2f}%')

In [ ]:
max_len = 30
below_threshold_len(max_len, X_train)

In [ ]:
def pad_sequences(sentences, max_len):
  features = np.zeros((len(sentences), max_len), dtype=int)
  for index, sentence in enumerate(sentences):
    if len(sentence) != 0:
      features[index, :len(sentence)] = np.array(sentence)[:max_len]
  return features

In [ ]:
padded_X_train = pad_sequences(encoded_X_train, max_len=max_len)
padded_X_valid = pad_sequences(encoded_X_valid, max_len=max_len)
padded_X_test = pad_sequences(encoded_X_test, max_len=max_len)

print('훈련 데이터의 크기 :', padded_X_train.shape)
print('검증 데이터의 크기 :', padded_X_valid.shape)
print('테스트 데이터의 크기 :', padded_X_test.shape)

In [ ]:
print('첫번째 샘플의 길이 :', len(padded_X_train[0]))
print('첫번째 샘플 :', padded_X_train[0])

# 2. LSTM을 이용한 네이버 영화 리뷰 분류 모델

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
train_label_tensor = torch.tensor(np.array(y_train))
valid_label_tensor = torch.tensor(np.array(y_valid))
test_label_tensor = torch.tensor(np.array(y_test))

In [ ]:
print(train_label_tensor[:10])

In [ ]:
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.xpu.is_available():
    device = "xpu"
device = torch.device(device)
print("사용 기기:", device)

### 2.1 LSTM 모델 클래스 구현
각 층의 출력의 크기를 이해하는 것이 중요!
- 인코딩된 문장이 입력으로 주어짐
    - 입력층 텐서의 크기(배치 크기, 문장 길이)의 크기를 가지는 텐서
- 임베딩층을 지나고 나면 입력 텐서의 각 단어가 임베딩 벡터로 변환
    - 임베딩 텐서의 크기: (배치 크기, 문장 길이, 임베딩 벡터의 차원)
- LSTM을 거쳐 순전파되고, 마지막 시점(timestep)의 은닉층 값이 출력층으로 전달
    - 마지막 은닉 텐서의 크기: (배치 크기, 은닉 상태의 차원)
- 출력층은 지난 결과는 소프트맥스 회귀를 수행
    - 출력 텐서의 크기: (배치 크기, 분류하고자하는 카테고리의 수)
- 출력 텐서를 배치 단위로 데이터 묶음을 꺼낼 수 있는 데이터로더로 전달

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)

        # LSTM은 (hidden state, cell state)의 튜플을 반환합니다
        lstm_out, (hidden, cell) = self.lstm(embedded)  # lstm_out: (batch_size, seq_length, hidden_dim), hidden: (1, batch_size, hidden_dim)

        last_hidden = hidden.squeeze(0)  # (batch_size, hidden_dim)
        logits = self.fc(last_hidden)  # (batch_size, output_dim)
        return logits

In [114]:
encoded_train = torch.tensor(padded_X_train).to(torch.int64)
train_dataset = torch.utils.data.TensorDataset(encoded_train, train_label_tensor)
train_dataloader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=64, num_workers = 2)

encoded_test = torch.tensor(padded_X_test).to(torch.int64)
test_dataset = torch.utils.data.TensorDataset(encoded_test, test_label_tensor)
test_dataloader = torch.utils.data.DataLoader(test_dataset, shuffle=True, batch_size=512, num_workers = 2)

encoded_valid = torch.tensor(padded_X_valid).to(torch.int64)
valid_dataset = torch.utils.data.TensorDataset(encoded_valid, valid_label_tensor)
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, shuffle=True, batch_size=512, num_workers = 2)

In [115]:
num_epochs = 5
total_batch = len(train_dataloader)
print('총 배치의 수 : {}'.format(total_batch)) # 116,324/32=3,635

총 배치의 수 : 1818


In [116]:
embedding_dim = 100
hidden_dim = 128
output_dim = 2
learning_rate = 0.01
num_epochs = 10

In [117]:
model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
model.to(device)

TextClassifier(
  (embedding): Embedding(19193, 100)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

In [118]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 2.2 평가 코드 작성
#### 2.21 정확도 평가 함수 작성

In [119]:
def calculate_accuracy(logits, labels):
    predicted = torch.argmax(logits, dim=1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total
    return accuracy

#### 2.22 모델 평가 함수 작성
- `model.eval()`: 모델을 평가 모드로 설정
    - 일부 레이어, 예를 들어 드롭아웃이나 배치 정규화는 학습과 평가 시 다르게 동작함
    - 평가 모드가 설정되지 않으면, 이로 인해 평가 결과가 제대로 나오지 않을 수 있음

- `with torch.no_grad()`: 자동 미분 엔진에서 기울기(gradient) 계산을 비활성화
    - 평가 중에는 기울기를 계산할 필요가 없음
    - 이 설정이 적용되지 않으면, 계산 및 메모리 리소스를 필요 이상으로 점유하게 됨

In [120]:
def evaluate(model, valid_dataloader, criterion, device):
    val_loss = 0
    val_correct = 0
    val_total = 0

    model.eval()
    with torch.no_grad():
        # 데이터로더로부터 배치 크기만큼의 데이터를 연속으로 로드
        for batch_X, batch_y in valid_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            # 모델의 예측값
            logits = model(batch_X)

            # 손실을 계산
            loss = criterion(logits, batch_y)

            # 정확도와 손실을 계산함
            val_loss += loss.item()
            val_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
            val_total += batch_y.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= len(valid_dataloader)

    return val_loss, val_accuracy

### 2.3 학습
- 딥러닝 모델을 훈련하고 검증하는 과정을 반복
- 검증 손실이 개선될 때마다 모델의 가중치를 저장
- 각 에포크마다 훈련 손실과 정확도를 계산하고, 검증 데이터로 모델을 평가
- 검증 손실이 가장 낮은 경우 해당 모델의 가중치를 파일로 저장

In [ ]:
# Training loop
best_val_loss = float('inf')

# Training loop
for epoch in range(num_epochs):
    # Training
    train_loss = 0
    train_correct = 0
    train_total = 0
    model.train()
    for batch_X, batch_y in train_dataloader:

        # batch_X.shape == (batch_size, max_len)
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        logits = model(batch_X)

        # Compute loss
        loss = criterion(logits, batch_y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate training accuracy and loss
        train_loss += loss.item()
        train_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
        train_total += batch_y.size(0)

    train_accuracy = train_correct / train_total
    train_loss /= len(train_dataloader)

    # Validation
    val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)

    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

    # 검증 손실이 최소일 때 체크포인트 저장
    if val_loss < best_val_loss:
        print(f'Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}. 체크포인트를 저장합니다.')
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'models/best_model_checkpoint_ltsm.pth')

Epoch 1/10:
Train Loss: 0.4894, Train Accuracy: 0.7370
Validation Loss: 0.3745, Validation Accuracy: 0.8312
Validation loss improved from inf to 0.3745. 체크포인트를 저장합니다.
Epoch 2/10:
Train Loss: 0.3303, Train Accuracy: 0.8558
Validation Loss: 0.3465, Validation Accuracy: 0.8487
Validation loss improved from 0.3745 to 0.3465. 체크포인트를 저장합니다.
Epoch 3/10:
Train Loss: 0.2723, Train Accuracy: 0.8857
Validation Loss: 0.3425, Validation Accuracy: 0.8536
Validation loss improved from 0.3465 to 0.3425. 체크포인트를 저장합니다.
Epoch 4/10:
Train Loss: 0.2188, Train Accuracy: 0.9119
Validation Loss: 0.3677, Validation Accuracy: 0.8533
Epoch 5/10:
Train Loss: 0.1690, Train Accuracy: 0.9351
Validation Loss: 0.3952, Validation Accuracy: 0.8488
Epoch 6/10:
Train Loss: 0.1262, Train Accuracy: 0.9538
Validation Loss: 0.4733, Validation Accuracy: 0.8424
Epoch 7/10:
Train Loss: 0.0945, Train Accuracy: 0.9668
Validation Loss: 0.5365, Validation Accuracy: 0.8437
Epoch 8/10:
Train Loss: 0.0721, Train Accuracy: 0.9752
Valida

### 2.4 모델 로드 및 평가
- test 데이터셋에 대한 결과를 바탕으로 하이퍼파라미터 수정하여 튜닝

In [ ]:
# 모델 로드
model.load_state_dict(torch.load('models/best_model_checkpoint_ltsm.pth'))

# 모델을 device에 올립니다.
model.to(device)

# 검증 데이터에 대한 정확도와 손실 계산
val_loss, val_accuracy = evaluate(model, valid_dataloader, criterion, device)

print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

Best model validation loss: 0.3423
Best model validation accuracy: 0.8536


In [123]:
# 테스트 데이터에 대한 정확도와 손실 계산
test_loss, test_accuracy = evaluate(model, test_dataloader, criterion, device)

print(f'Best model test loss: {test_loss:.4f}')
print(f'Best model test accuracy: {test_accuracy:.4f}')

Best model test loss: 0.3505
Best model test accuracy: 0.8448


### 2.5 모델 테스트

In [124]:
index_to_tag = {0 : '부정', 1 : '긍정'}

In [125]:
def predict(text, model, word_to_index, index_to_tag):
    # Set the model to evaluation mode
    model.eval()

    # Tokenize the input text
    tokens = mecab.morphs(text) # 토큰화
    tokens = [word for word in tokens if not word in stopwords] # 불용어 제거
    token_indices = [word_to_index.get(token, 1) for token in tokens]

    # Convert tokens to tensor
    input_tensor = torch.tensor([token_indices], dtype=torch.long).to(device)  # (1, seq_length)

    # Pass the input tensor through the model
    with torch.no_grad():
        logits = model(input_tensor)  # (1, output_dim)

    # Get the predicted class index
    predicted_index = torch.argmax(logits, dim=1)

    # Convert the predicted index to its corresponding tag
    predicted_tag = index_to_tag[predicted_index.item()]

    return predicted_tag

In [126]:
test_input = "이 영화 개꿀잼 ㅋㅋㅋ"

predict(test_input, model, word_to_index, index_to_tag)

'긍정'

In [127]:
test_input = "이딴게 영화냐 ㅉㅉ"

predict(test_input, model, word_to_index, index_to_tag)

'부정'

In [128]:
test_input = "감독 뭐하는 놈이냐?"

predict(test_input, model, word_to_index, index_to_tag)

'부정'

In [129]:
test_input = "와 개쩐다 정말 세계관 최강자들의 영화다"

predict(test_input, model, word_to_index, index_to_tag)

'긍정'